In [11]:
#Nagy - Urine

In [12]:
import pandas as pd

# 文件路径
data_week0_file = "raw_data/Nagy/RCT_Urine/data/NAGY RCT priority specimens shipped Sept. 2022 v2022-11-22.xlsx"
aliquot_file = "aliquot_DCC_data_release_v2-0-5_shipped_updated-labs.tsv"
output_file = "molecular-test_nagy-urine_DCC_data_release_v2-0-7.tsv"
qc_file = "QC_molecular-test_nagy-urine_DCC_data_release_v2-0-7.tsv"

# 读取数据
df_data = pd.read_excel(data_week0_file, sheet_name="RCT Samples", header=1, dtype=str)
df_aliquot = pd.read_csv(aliquot_file, sep='\t', dtype=str)

# 清理列名空格
df_data.columns = df_data.columns.str.strip()
df_aliquot.columns = df_aliquot.columns.str.strip()

# 只保留Barcode存在的记录
df_data = df_data[df_data["Barcode"].notna()]
df_aliquot = df_aliquot[df_aliquot["*submitter_id"].notna()]

# 设置索引以提高查找效率
aliquot_lookup = df_aliquot.set_index("*submitter_id")

# 存储输出行和 QC 行
output_rows = []
qc_rows = []

# 定义要处理的生物标志物
biomarkers = [
    {"column": "Creatinine (mg/dL)", "lab_test": "Urine Creatinine", "unit": "mg/dL"},
    {"column": "KIM-1 (ng/mL)", "lab_test": "Urine KIM-1", "unit": "ng/mL"},
    {"column": "IL-18 (pg/mL)", "lab_test": "Urine IL-18", "unit": "pg/mL"},
    {"column": "L-FABP (pg/mL)", "lab_test": "Urine L-FABP", "unit": "pg/mL"},
    {"column": "NGAL (ng/mL)", "lab_test": "Urine NGAL", "unit": "ng/mL"},
]

# 遍历每一行 Barcode
for _, row in df_data.iterrows():
    barcode = row["Barcode"]
    if barcode in aliquot_lookup.index:
        ali_row = aliquot_lookup.loc[barcode]
        followup_id = ali_row["*follow_ups.submitter_id"]

        for biomarker in biomarkers:
            test_value = row.get(biomarker["column"], "")
            if pd.notna(test_value) and test_value != "":
                new_submitter_id = f"{followup_id}_{biomarker['lab_test'].split()[-1].replace('-', '')}"

                output_rows.append({
                    "*type": "molecular_test",
                    "project_id": "ARDaC-AlcHepNet",
                    "*submitter_id": new_submitter_id,
                    "aliquots.submitter_id": barcode,
                    "*follow_ups.submitter_id": followup_id,
                    "blood_test_normal_range_lower": "",
                    "blood_test_normal_range_upper": "",
                    "days_to_test": "",
                    "gene_symbol": "",
                    "laboratory_test": biomarker["lab_test"],
                    "molecular_analysis_method": "",
                    "test_result": "",
                    "test_unit": biomarker["unit"],
                    "test_value": test_value if test_value != "ND" else ""
                })
    else:
        qc_rows.append({
            "source_file": "NAGY RCT priority specimens shipped Sept. 2022.xlsx",
            "Barcode": barcode
        })

# 保存输出文件
pd.DataFrame(output_rows).to_csv(output_file, sep="\t", index=False)
pd.DataFrame(qc_rows).to_csv(qc_file, sep="\t", index=False)

In [13]:
# Mehal - Blood ACE

In [14]:
import pandas as pd

# 文件路径
data_week0_file = "raw_data/Mehal/RCT_Blood/data/Aki_ELISA_ACE_Final_Data[1].xlsx"
aliquot_file = "aliquot_DCC_data_release_v2-0-5_shipped_updated-labs.tsv"
output_file = "molecular-test_mehal-ace_DCC_data_release_v2-0-7.tsv"
qc_file = "QC_molecular-test_mehal-ace_DCC_data_release_v2-0-7.tsv"

# 读取数据（注意 header=1 表示从第2行作为表头）
df_data = pd.read_excel(data_week0_file, sheet_name="with barcodes", header=1, dtype=str)
df_aliquot = pd.read_csv(aliquot_file, sep='\t', dtype=str)

# 清理列名空格
df_data.columns = df_data.columns.str.strip()
df_aliquot.columns = df_aliquot.columns.str.strip()

# 确保关键列不为空
df_data = df_data[df_data["Barcode"].notna()]
df_aliquot = df_aliquot[df_aliquot["*submitter_id"].notna()]

# 设置索引以加快查找速度
aliquot_lookup = df_aliquot.set_index("*submitter_id")

# 存储输出和QC行
output_rows = []
qc_rows = []

# 遍历每一行Barcode
for _, row in df_data.iterrows():
    barcode = row["Barcode"]
    if barcode in aliquot_lookup.index:
        ali_row = aliquot_lookup.loc[barcode]
        followup_id = ali_row["*follow_ups.submitter_id"]
        new_submitter_id = f"{followup_id}_ACE"

        output_rows.append({
            "*type": "molecular_test",
            "project_id": "ARDaC-AlcHepNet",
            "*submitter_id": new_submitter_id,
            "aliquots.submitter_id": barcode,
            "*follow_ups.submitter_id": followup_id,
            "blood_test_normal_range_lower": "",
            "blood_test_normal_range_upper": "",
            "days_to_test": "",
            "gene_symbol": "",
            "laboratory_test": "ACE",
            "molecular_analysis_method": "",
            "test_result": "",
            "test_unit": "µg/l",
            "test_value": row.get("Plasma ACE Conc (µg/L)", "")
        })
    else:
        qc_rows.append({
            "source_file": "Aki_ELISA_ACE_Final_Data[1].xlsx",
            "Barcode": barcode
        })

# 保存结果
output_df = pd.DataFrame(output_rows)
output_df.to_csv(output_file, sep="\t", index=False)

qc_df = pd.DataFrame(qc_rows)
qc_df.to_csv(qc_file, sep="\t", index=False)

In [15]:
# Mehal - Blood Renin

In [16]:
import pandas as pd

# 文件路径
data_week0_file = "raw_data/Mehal/RCT_Blood/data/Aki_ELISA_RENIN_Final_Data[1].xlsx"
aliquot_file = "aliquot_DCC_data_release_v2-0-5_shipped_updated-labs.tsv"
output_file = "molecular-test_mehal-renin_DCC_data_release_v2-0-7.tsv"
qc_file = "QC_molecular-test_mehal-renin_DCC_data_release_v2-0-7.tsv"

# 读取数据（注意 header=1 表示从第2行作为表头）
df_data = pd.read_excel(data_week0_file, sheet_name="with barcodes", header=1, dtype=str)
df_aliquot = pd.read_csv(aliquot_file, sep='\t', dtype=str)

# 清理列名空格
df_data.columns = df_data.columns.str.strip()
df_aliquot.columns = df_aliquot.columns.str.strip()

# 确保关键列不为空
df_data = df_data[df_data["Barcode"].notna()]
df_aliquot = df_aliquot[df_aliquot["*submitter_id"].notna()]

# 设置索引以加快查找速度
aliquot_lookup = df_aliquot.set_index("*submitter_id")

# 存储输出和QC行
output_rows = []
qc_rows = []

# 遍历每一行Barcode
for _, row in df_data.iterrows():
    barcode = row["Barcode"]
    if barcode in aliquot_lookup.index:
        ali_row = aliquot_lookup.loc[barcode]
        followup_id = ali_row["*follow_ups.submitter_id"]
        new_submitter_id = f"{followup_id}_Renin"

        output_rows.append({
            "*type": "molecular_test",
            "project_id": "ARDaC-AlcHepNet",
            "*submitter_id": new_submitter_id,
            "aliquots.submitter_id": barcode,
            "*follow_ups.submitter_id": followup_id,
            "blood_test_normal_range_lower": "",
            "blood_test_normal_range_upper": "",
            "days_to_test": "",
            "gene_symbol": "",
            "laboratory_test": "Renin",
            "molecular_analysis_method": "",
            "test_result": "",
            "test_unit": "ng/mL",
            "test_value": row.get("Plasma Renin Conc (ng/mL)", "")
        })
    else:
        qc_rows.append({
            "source_file": "Aki_ELISA_RENIN_Final_Data[1].xlsx",
            "Barcode": barcode
        })

# 保存结果
output_df = pd.DataFrame(output_rows)
output_df.to_csv(output_file, sep="\t", index=False)

qc_df = pd.DataFrame(qc_rows)
qc_df.to_csv(qc_file, sep="\t", index=False)

In [17]:
#Liangpunsakul - ORM1

In [18]:
import pandas as pd

# 文件路径
data_week0_file = "raw_data/Liangpunsakul/Data week 0.xlsx"
aliquot_file = "aliquot_DCC_data_release_v2-0-5_shipped_updated-labs.tsv"
output_file = "molecular-test_liangpunsakul-orm1_DCC_data_release_v2-0-7.tsv"
qc_file = "QC_molecular-test_liangpunsakul-orm1_DCC_data_release_v2-0-7.tsv"

# 读取数据
df_data = pd.read_excel(data_week0_file, dtype=str)
df_aliquot = pd.read_csv(aliquot_file, sep='\t', dtype=str)

# 确保关键列不为空
df_data = df_data[df_data["Barcode"].notna()]
df_aliquot = df_aliquot[df_aliquot["*submitter_id"].notna()]

# 设置索引以加快查找速度
aliquot_lookup = df_aliquot.set_index("*submitter_id")

# 存储输出和QC行
output_rows = []
qc_rows = []

# 遍历每一行Barcode
for _, row in df_data.iterrows():
    barcode = row["Barcode"]
    if barcode in aliquot_lookup.index:
        ali_row = aliquot_lookup.loc[barcode]
        followup_id = ali_row["*follow_ups.submitter_id"]
        new_submitter_id = f"{followup_id}_ORM1"

        output_rows.append({
            "*type": "molecular_test",
            "project_id": "ARDaC-AlcHepNet",
            "*submitter_id": new_submitter_id,
            "aliquots.submitter_id": barcode,
            "*follow_ups.submitter_id": followup_id,
            "blood_test_normal_range_lower": "",
            "blood_test_normal_range_upper": "",
            "days_to_test": "",
            "gene_symbol": "",
            "laboratory_test": "ORM1",
            "molecular_analysis_method": "",
            "test_result": "",
            "test_unit": "µg/ml",
            "test_value": row.get("ORM1µgml", "")
        })
    else:
        qc_rows.append({
            "source_file": "Data week 0.xlsx",
            "Barcode": barcode
        })

# 保存结果
output_df = pd.DataFrame(output_rows)
output_df.to_csv(output_file, sep="\t", index=False)

qc_df = pd.DataFrame(qc_rows)
qc_df.to_csv(qc_file, sep="\t", index=False)

In [19]:
import pandas as pd

# 文件路径
data_week0_file = "raw_data/Liangpunsakul/ORM1_more samples with barcode.xlsx"
aliquot_file = "aliquot_DCC_data_release_v2-0-5_shipped_updated-labs.tsv"
output_file = "molecular-test_liangpunsakul-orm1_part2_DCC_data_release_v2-0-7.tsv"
qc_file = "QC_molecular-test_liangpunsakul-orm1_part2_DCC_data_release_v2-0-7.tsv"

# 读取数据
df_data = pd.read_excel(data_week0_file, dtype=str)
df_aliquot = pd.read_csv(aliquot_file, sep='\t', dtype=str)

# 确保关键列不为空
df_data = df_data[df_data["Barcode"].notna()]
df_aliquot = df_aliquot[df_aliquot["*submitter_id"].notna()]

# 设置索引以加快查找速度
aliquot_lookup = df_aliquot.set_index("*submitter_id")

# 存储输出和QC行
output_rows = []
qc_rows = []

# 遍历每一行Barcode
for _, row in df_data.iterrows():
    barcode = row["Barcode"]
    if barcode in aliquot_lookup.index:
        ali_row = aliquot_lookup.loc[barcode]
        followup_id = ali_row["*follow_ups.submitter_id"]
        new_submitter_id = f"{followup_id}_ORM1"

        output_rows.append({
            "*type": "molecular_test",
            "project_id": "ARDaC-AlcHepNet",
            "*submitter_id": new_submitter_id,
            "aliquots.submitter_id": barcode,
            "*follow_ups.submitter_id": followup_id,
            "blood_test_normal_range_lower": "",
            "blood_test_normal_range_upper": "",
            "days_to_test": "",
            "gene_symbol": "",
            "laboratory_test": "ORM1",
            "molecular_analysis_method": "",
            "test_result": "",
            "test_unit": "µg/ml",
            "test_value": row.get("ORM1µgml", "")
        })
    else:
        qc_rows.append({
            "source_file": "ORM1_more samples with barcode.xlsx",
            "Barcode": barcode
        })

# 保存结果
output_df = pd.DataFrame(output_rows)
output_df.to_csv(output_file, sep="\t", index=False)

qc_df = pd.DataFrame(qc_rows)
qc_df.to_csv(qc_file, sep="\t", index=False)

In [ ]:
import pandas as pd

# 文件路径
data_week0_file = "raw_data/Liangpunsakul/ORM1_more samples with barcode.xlsx"
aliquot_file = "aliquot_DCC_data_release_v2-0-5_shipped_updated-labs.tsv"
output_file = "molecular-test_liangpunsakul-orm1_part2_DCC_data_release_v2-0-7.tsv"
qc_file = "QC_molecular-test_liangpunsakul-orm1_part2_DCC_data_release_v2-0-7.tsv"

# 读取数据
df_data = pd.read_excel(data_week0_file, dtype=str)
df_aliquot = pd.read_csv(aliquot_file, sep='\t', dtype=str)

# 确保关键列不为空
df_data = df_data[df_data["Barcode"].notna()]
df_aliquot = df_aliquot[df_aliquot["*submitter_id"].notna()]

# 设置索引以加快查找速度
aliquot_lookup = df_aliquot.set_index("*submitter_id")

# 存储输出和QC行
output_rows = []
qc_rows = []

# 遍历每一行Barcode
for _, row in df_data.iterrows():
    barcode = row["Barcode"]
    if barcode in aliquot_lookup.index:
        ali_row = aliquot_lookup.loc[barcode]
        followup_id = ali_row["*follow_ups.submitter_id"]
        new_submitter_id = f"{followup_id}_ORM1"

        output_rows.append({
            "*type": "molecular_test",
            "project_id": "ARDaC-AlcHepNet",
            "*submitter_id": new_submitter_id,
            "aliquots.submitter_id": barcode,
            "*follow_ups.submitter_id": followup_id,
            "blood_test_normal_range_lower": "",
            "blood_test_normal_range_upper": "",
            "days_to_test": "",
            "gene_symbol": "",
            "laboratory_test": "ORM1",
            "molecular_analysis_method": "",
            "test_result": "",
            "test_unit": "µg/ml",
            "test_value": row.get("ORM1µgml", "")
        })
    else:
        qc_rows.append({
            "source_file": "ORM1_more samples with barcode.xlsx",
            "Barcode": barcode
        })

# 保存结果
output_df = pd.DataFrame(output_rows)
output_df.to_csv(output_file, sep="\t", index=False)

qc_df = pd.DataFrame(qc_rows)
qc_df.to_csv(qc_file, sep="\t", index=False)

In [1]:
# Schnabl
import pandas as pd

# 文件路径
data_week0_file = "raw_data/Schnabl/AhR activity value AlcHepNet.xlsx"
aliquot_file = "aliquot_DCC_data_release_v2-0-5_shipped_updated-labs.tsv"
output_file = "molecular-test_schnabl_DCC_data_release_v2-0-7.tsv"
qc_file = "QC_molecular-test_schnabl_DCC_data_release_v2-0-7.tsv"

# 读取数据
df_data = pd.read_excel(data_week0_file, dtype=str)
df_aliquot = pd.read_csv(aliquot_file, sep='\t', dtype=str)

# 确保关键列不为空
df_data = df_data[df_data["SUBJECT_ID"].notna()]
df_aliquot = df_aliquot[df_aliquot["*submitter_id"].notna()]

# 设置索引以加快查找速度
aliquot_lookup = df_aliquot.set_index("*submitter_id")

# 存储输出和QC行
output_rows = []
qc_rows = []

# 遍历每一行Barcode
for _, row in df_data.iterrows():
    barcode = row["SUBJECT_ID"]
    if barcode in aliquot_lookup.index:
        ali_row = aliquot_lookup.loc[barcode]
        followup_id = ali_row["*follow_ups.submitter_id"]
        new_submitter_id = f"{followup_id}_schnabl"

        output_rows.append({
            "*type": "molecular_test",
            "project_id": "ARDaC-AlcHepNet",
            "*submitter_id": new_submitter_id,
            "aliquots.submitter_id": barcode,
            "*follow_ups.submitter_id": followup_id,
            "blood_test_normal_range_lower": "",
            "blood_test_normal_range_upper": "",
            "days_to_test": "",
            "gene_symbol": "",
            "laboratory_test": "",
            "molecular_analysis_method": "",
            "test_result": "",
            "test_unit": "",
            "test_value": row.get("AhR activity", "")
        })
    else:
        qc_rows.append({
            "source_file": "AhR activity value AlcHepNet.xlsx",
            "Barcode": barcode
        })

# 保存结果
output_df = pd.DataFrame(output_rows)
output_df.to_csv(output_file, sep="\t", index=False)

qc_df = pd.DataFrame(qc_rows)
qc_df.to_csv(qc_file, sep="\t", index=False)
